In [2]:
import numpy as np
import pandas as pd
import torch

# Configs:
DATASET = "./ASL_Citizen"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_SIGNS = 2731


print(f"Using: {DEVICE}")

Using: cuda


In [3]:
import os
import pandas as pd

# Collect all unique glosses from every split
all_glosses = set()

for split in ["train", "val", "test"]:
    df = pd.read_csv(f"{DATASET}/splits/{split}.csv")
    all_glosses.update(df["Gloss"].unique())

# Create one consistent mapping for all splits
gloss_to_label = {
    gloss: idx
    for idx, gloss in enumerate(sorted(all_glosses))
}

print(f"Total unique glosses: {len(gloss_to_label)}")
assert(len(gloss_to_label) == NUM_SIGNS)

def add_label(subset: str) -> None:
    df = pd.read_csv(f"{DATASET}/splits/{subset}.csv")

    print(f"\n{subset} original:")
    print(df.head())

    # (Optional) sort by Gloss
    df = df.sort_values("Gloss")

    # Use the global mapping
    df["Label"] = df["Gloss"].map(gloss_to_label)

    print(f"{subset} with labels:")
    print(df.head())

    output_path = "data/splits"
    os.makedirs(output_path, exist_ok=True)

    df.to_csv(f"{output_path}/{subset}.csv", index=False)


for split in ["train", "val", "test"]:
    add_label(split)


Total unique glosses: 2731

train original:
  Participant ID                        Video file       Gloss ASL-LEX Code
0             P1       15890366051589533-APPLE.mp4       APPLE     A_03_054
1             P1  35618482303951104-IMPOSSIBLE.mp4  IMPOSSIBLE     B_01_032
2             P1         6958143575951994-PARK.mp4        PARK     E_03_028
3             P1     8006032738002744-SOCCER 2.mp4     SOCCER2     F_03_032
4             P1       37542279833186454-STINK.mp4       STINK     H_01_064
train with labels:
      Participant ID                      Video file    Gloss ASL-LEX Code  \
23655            P31   3827306090663467-1 DOLLAR.mp4  1DOLLAR     C_02_025   
4353             P37  16792698524451422-1 DOLLAR.mp4  1DOLLAR     C_02_025   
33798            P11   6868778695018762-1 DOLLAR.mp4  1DOLLAR     C_02_025   
30480            P11   6870709051348651-1 DOLLAR.mp4  1DOLLAR     C_02_025   
14998            P50   0719792557216079-1 DOLLAR.mp4  1DOLLAR     C_02_025   

       Label

In [4]:
class Stats:
    """Accumulates loss and ranking metrics (Recall@1, Recall@5,
    Recall@10, MRR, DCG) over an epoch and reports their averages."""

    def __init__(self):
        self.losses = []
        self.recall_at_1 = []
        self.recall_at_5 = []
        self.recall_at_10 = []
        self.mrr = []
        self.dcg = []

    def update(self, loss: float, model_output: torch.Tensor, labels: torch.Tensor):
        ranking = torch.argsort(model_output, dim=1, descending=True)
        ranks = (ranking == labels.unsqueeze(1)).nonzero(as_tuple=True)[1] + 1
        ranks = ranks.to(dtype=torch.float32)

        self.losses.append(loss)
        self.recall_at_1.append((ranks <= 1).to(dtype=torch.float32).mean().item())
        self.recall_at_5.append((ranks <= 5).to(dtype=torch.float32).mean().item())
        self.recall_at_10.append((ranks <= 10).to(dtype=torch.float32).mean().item())
        self.mrr.append((1.0 / ranks).mean().item())
        self.dcg.append((1.0 / torch.log2(ranks + 1)).mean().item())

    def compute(self) -> dict[str, float]:
        if not self.losses:
            return {}
        
        return {
            "loss": sum(self.losses) / len(self.losses),
            "recall@1": sum(self.recall_at_1) / len(self.recall_at_1),
            "recall@5": sum(self.recall_at_5) / len(self.recall_at_5),
            "recall@10": sum(self.recall_at_10) / len(self.recall_at_10),
            "mrr": sum(self.mrr) / len(self.mrr),
            "dcg": sum(self.dcg) / len(self.dcg),
        }


In [5]:
from torchcodec.decoders import VideoDecoder
from torch.utils.data import Dataset


class ASLCitizen(Dataset):
    def __init__(self, 
                 annotations_file: str, 
                 data_dir: str, 
                 image_processor,       
                 num_frames: int = 16,  
                 transforms=None, 
                 apply_transforms=True):
        """
        :param annotations_file: the path of the .csv file containing file paths and labels
        :param data_dir: the path of the directory holding the .npy files that represent the videos
        :param transforms: transforms on the asl_citizen, defaulted to None
        """
        self.annotations = pd.read_csv(annotations_file)
        self.data_dir = data_dir
        self.image_processor = image_processor
        self.num_frames = num_frames
        self.transforms = transforms
        self.image_processor = image_processor
        self.apply_transforms = apply_transforms
        self.file_names = self.annotations["Video file"].to_list()
        self.labels = self.annotations["Label"].to_list()
        
    def __len__(self):
        return len(self.annotations)

    def __sample_frame_indices(self, total_frames: int) -> np.ndarray:
        """Uniformly sample `num_frames` indices across the whole clip.
        Pads by repeating the last frame if the clip is shorter than
        num_frames, so every sample has the exact same temporal length."""
        if total_frames >= self.num_frames:
            indices = np.linspace(0, total_frames - 1, self.num_frames)
            return np.round(indices).astype(int)
        indices = np.arange(total_frames)
        pad = self.num_frames - total_frames
        return np.concatenate([indices, np.repeat(indices[-1], pad)])
        
    def __getitem__(self, idx):
        data_path = os.path.join(self.data_dir, self.file_names[idx])
        label = self.labels[idx]

        decoder = VideoDecoder(data_path)
        total_frames = decoder.metadata.num_frames
        frame_indices = self.__sample_frame_indices(total_frames)
        

        frames = decoder.get_frames_at(indices=frame_indices.tolist()).data
 
        if self.apply_transforms:
            # VideoMAEImageProcessor expects a list of HWC frames (numpy or PIL).
            frames_hwc = [f.permute(1, 2, 0).numpy() for f in frames]
            processed = self.image_processor(frames_hwc, return_tensors="pt")
            pixel_values = processed["pixel_values"].squeeze(0)  # (T, C, H, W)
        else:
            pixel_values = frames.float() / 255.0
 
        return pixel_values, label

In [6]:
import torch
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

class ModelTrainer:
    def __init__(self, model, optimizer, loss_fn, train_dataloader, val_dataloader, epochs):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader

        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=0.01,
            steps_per_epoch=len(self.train_dataloader),
            epochs=epochs,
        )

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model.to(device=self.device)

    # code based on https://github.com/ml-jku/hopfield-layers/tree/master
    def train_model(self, num_epochs) -> tuple[dict, dict]:
        train_history, val_history = {}, {}
        for epoch in range(num_epochs):
            train_stats = self.__train_epoch()
            val_stats = self.__eval()

            for key, value in train_stats.items():
                train_history.setdefault(key, []).append(value)
            for key, value in val_stats.items():
                val_history.setdefault(key, []).append(value)

            print(f"epoch {epoch + 1}:")
            print(f"\ttrain: {train_stats}")
            print(f"\tval: {val_stats}")

        return train_history, val_history

    # code based on https://github.com/ml-jku/hopfield-layers/tree/master
    def __train_epoch(self) -> dict[str, float]:
        self.model.train()
        stats = Stats()
        for batch in tqdm(self.train_dataloader):
            data, labels = batch
            data, labels = data.to(self.device), labels.to(self.device)

            # Model forward propagation
            model_output = self.model.forward(input=data.to(dtype=torch.float64))

            # Update model parameters
            self.optimizer.zero_grad()
            loss = self.loss_fn(model_output, labels.to(dtype=torch.int64))
            loss.backward()
            clip_grad_norm_(
                parameters=self.model.parameters(), max_norm=1.0, norm_type=2
            )
            self.optimizer.step()
            self.scheduler.step()

            # Compute performance measures of current model.
            stats.update(loss.detach().item(), model_output.detach(), labels)

        # Report progress of training procedure
        return stats.compute()

    # code based on https://github.com/ml-jku/hopfield-layers/tree/master
    def __eval(self) -> dict[str, float]:
        self.model.eval()
        with torch.no_grad():
            stats = Stats()
            for batch in tqdm(self.val_dataloader):
                data, labels = batch
                data, labels = data.to(self.device), labels.to(self.device)

                # Model forward propagation
                model_output = self.model.forward(input=data.to(dtype=torch.float64))
                loss = self.loss_fn(model_output, labels.to(dtype=torch.int64))

                # Compute performance measures of current model
                stats.update(loss.detach().item(), model_output.detach(), labels)

            # Report results on validation set
            return stats.compute()

In [7]:
!pip install -q "transformers>=4.45,<5.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00


In [8]:
import torch

from transformers import (
    VideoMAEImageProcessor,
    AutoConfig,
    AutoModel,
)


def load_videomae_v2(checkpoint: str = "OpenGVLab/VideoMAEv2-Base"):
    processor = VideoMAEImageProcessor.from_pretrained(checkpoint)
    config = AutoConfig.from_pretrained(checkpoint, trust_remote_code=True)
    encoder = AutoModel.from_pretrained(
        checkpoint, config=config, trust_remote_code=True
    )
    return processor, encoder


class FineTuningClassifier(torch.nn.Module):
    def __init__(self, encoder: torch.nn.Module, hidden_size: int, num_signs: int):
        super().__init__()
        self.encoder = encoder
        self.head = torch.nn.Linear(hidden_size, num_signs)

    def forward(self, pixel_values):
        pixel_values = pixel_values.permute(0, 2, 1, 3, 4)
        pooled = self.encoder.extract_features(pixel_values)
        return self.head(pooled)


def load_model():
    _, encoder_v2 = load_videomae_v2("OpenGVLab/VideoMAEv2-Base")
    hidden_size = encoder_v2.config.model_config["embed_dim"]
    return FineTuningClassifier(
        encoder_v2, hidden_size=hidden_size, num_signs=2731
    )


In [9]:
import numpy as np
from torchvision.models.video import mvit_v2_s, MViT_V2_S_Weights


class MViTImageProcessor:
    def __init__(self, weights=MViT_V2_S_Weights.KINETICS400_V1):
        self.transform = weights.transforms()

    def __call__(self, frames_hwc, return_tensors="pt"):
        frames = torch.from_numpy(np.stack(frames_hwc)).permute(0, 3, 1, 2)  # (T, C, H, W) uint8
        pixel_values = self.transform(frames)  # -> (C, T, H, W), resized/cropped/normalized
        return {"pixel_values": pixel_values.unsqueeze(0)}  # fake batch dim, like HF processors


class FineTuningClassifierMViT(torch.nn.Module):
    def __init__(self, encoder: torch.nn.Module, hidden_size: int, num_signs: int):
        super().__init__()
        self.encoder = encoder
        self.head = torch.nn.Linear(hidden_size, num_signs)

    def forward(self, pixel_values):
        pooled = self.encoder(pixel_values)  # encoder.head is Identity, so this is pooled features
        return self.head(pooled)


def load_model_mvit():
    weights = MViT_V2_S_Weights.KINETICS400_V1
    encoder = mvit_v2_s(weights=weights)
    hidden_size = encoder.head[-1].in_features  # grab before replacing
    encoder.head = torch.nn.Identity()
    processor = MViTImageProcessor(weights)
    return FineTuningClassifierMViT(encoder, hidden_size, num_signs=2731), processor

In [10]:
from torch.utils.data import Dataset, DataLoader


def collate_fn(batch):
    """Stacks a list of (T, C, H, W) samples into (B, T, C, H, W)."""
    pixel_values = torch.stack([item[0] for item in batch], dim=0)
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    return pixel_values, labels

In [11]:
class LPFTTrainer:
    """
    Runs linear probing followed by full fine-tuning on a
    FineTuningClassifier(encoder, head) model. Saves the best-performing
    checkpoint (by validation metric) seen across both phases.

    Usage:
        trainer = LPFTTrainer(
            model, loss_fn, train_dataloader, val_dataloader,
            checkpoint_path="best_model.pt", checkpoint_metric="recall@1",
        )
        history = trainer.run(
            lp_epochs=5, lp_lr=1e-3,
            ft_epochs=10, ft_lr=2e-5,
        )
    """

    def __init__(
        self,
        model,
        loss_fn,
        train_dataloader,
        val_dataloader,
        use_amp=True,
        checkpoint_path: str | None = "best_model.pt",
        checkpoint_metric: str = "recall@1",
        checkpoint_mode: str = "max",  # "max" for recall/mrr/dcg, "min" for loss
    ):
        self.model = model
        self.loss_fn = loss_fn
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.use_amp = use_amp and self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler(enabled=self.use_amp)

        assert checkpoint_mode in ("max", "min")
        self.checkpoint_path = checkpoint_path
        self.checkpoint_metric = checkpoint_metric
        self.checkpoint_mode = checkpoint_mode
        self.best_metric = float("-inf") if checkpoint_mode == "max" else float("inf")

    # -- public API ----------------------------------------------------
    def run(
        self,
        lp_epochs: int,
        lp_lr: float,
        ft_epochs: int,
        ft_lr: float,
        head_lr_in_ft: float | None = None,
        max_norm: float = 1.0,
    ) -> dict[str, dict]:
        """
        :param lp_epochs: number of linear-probe epochs (encoder frozen)
        :param lp_lr: LR for the head during linear probing
        :param ft_epochs: number of full fine-tuning epochs (encoder unfrozen)
        :param ft_lr: LR for the encoder during fine-tuning
        :param head_lr_in_ft: optional separate (usually slightly higher) LR
            for the head during fine-tuning. Defaults to ft_lr if None.
        :param max_norm: gradient clipping norm, applied in both phases
        :return: {"linear_probe": {...}, "fine_tune": {...}} each holding
                 (train_history, val_history) tuples
        """
        history = {}

        print("=" * 60)
        print(f"Phase 1: Linear probing ({lp_epochs} epochs, lr={lp_lr})")
        print("=" * 60)
        history["linear_probe"] = self.__linear_probe(lp_epochs, lp_lr, max_norm)

        print("=" * 60)
        print(f"Phase 2: Full fine-tuning ({ft_epochs} epochs, lr={ft_lr})")
        print("=" * 60)
        head_lr_in_ft = ft_lr if head_lr_in_ft is None else head_lr_in_ft
        history["fine_tune"] = self.__fine_tune(ft_epochs, ft_lr, head_lr_in_ft, max_norm)

        if self.checkpoint_path is not None:
            print("=" * 60)
            print(
                f"Best {self.checkpoint_metric}={self.best_metric:.4f} "
                f"saved to {self.checkpoint_path}"
            )
            print("=" * 60)

        return history

    def load_best_checkpoint(self) -> dict:
        """Loads the best-saved checkpoint's weights back into self.model
        and returns the full checkpoint dict (with phase/epoch/metric info)."""
        checkpoint = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        return checkpoint

    # -- phase 1: linear probe -------------------------------------------
    def __linear_probe(self, epochs: int, lr: float, max_norm: float):
        for p in self.model.encoder.parameters():
            p.requires_grad = False
        for p in self.model.head.parameters():
            p.requires_grad = True

        optimizer = torch.optim.AdamW(self.model.head.parameters(), lr=lr)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            steps_per_epoch=len(self.train_dataloader),
            epochs=epochs,
        )

        return self.__run_epochs(
            epochs, optimizer, scheduler, max_norm,
            encoder_train_mode=False, phase_name="linear_probe",
        )

    # -- phase 2: full fine-tune -----------------------------------------
    def __fine_tune(self, epochs: int, encoder_lr: float, head_lr: float, max_norm: float):
        for p in self.model.parameters():
            p.requires_grad = True

        optimizer = torch.optim.AdamW(
            [
                {"params": self.model.encoder.parameters(), "lr": encoder_lr},
                {"params": self.model.head.parameters(), "lr": head_lr},
            ]
        )
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[encoder_lr, head_lr],
            steps_per_epoch=len(self.train_dataloader),
            epochs=epochs,
        )

        return self.__run_epochs(
            epochs, optimizer, scheduler, max_norm,
            encoder_train_mode=True, phase_name="fine_tune",
        )

    # -- shared epoch loop -------------------------------------------------
    def __run_epochs(self, epochs, optimizer, scheduler, max_norm, encoder_train_mode, phase_name):
        train_history, val_history = {}, {}
        for epoch in range(epochs):
            train_stats = self.__train_epoch(optimizer, scheduler, max_norm, encoder_train_mode)
            val_stats = self.__eval()

            for key, value in train_stats.items():
                train_history.setdefault(key, []).append(value)
            for key, value in val_stats.items():
                val_history.setdefault(key, []).append(value)

            print(f"epoch {epoch + 1}/{epochs}:")
            print(f"\ttrain: {train_stats}")
            print(f"\tval: {val_stats}")

            self.__maybe_save_checkpoint(val_stats, phase_name, epoch)

        return train_history, val_history

    def __maybe_save_checkpoint(self, val_stats: dict, phase_name: str, epoch: int):
        if self.checkpoint_path is None or self.checkpoint_metric not in val_stats:
            return

        value = val_stats[self.checkpoint_metric]
        improved = (
            value > self.best_metric if self.checkpoint_mode == "max"
            else value < self.best_metric
        )
        if improved:
            self.best_metric = value
            torch.save(
                {
                    "model_state_dict": self.model.state_dict(),
                    "phase": phase_name,
                    "epoch": epoch,
                    "val_stats": val_stats,
                },
                self.checkpoint_path,
            )
            print(f"\t-> new best {self.checkpoint_metric}={value:.4f}, saved to {self.checkpoint_path}")

    def __train_epoch(self, optimizer, scheduler, max_norm, encoder_train_mode) -> dict[str, float]:
        self.model.head.train()
        self.model.encoder.train(mode=encoder_train_mode)

        stats = Stats()
        for batch in tqdm(self.train_dataloader):
            data, labels = batch
            data = data.to(self.device, dtype=torch.float32)
            labels = labels.to(self.device, dtype=torch.int64)

            optimizer.zero_grad()
            with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                model_output = self.model(pixel_values=data)
                loss = self.loss_fn(model_output, labels)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(optimizer)
            clip_grad_norm_(
                parameters=[p for p in self.model.parameters() if p.requires_grad],
                max_norm=max_norm,
                norm_type=2,
            )
            self.scaler.step(optimizer)
            self.scaler.update()
            scheduler.step()

            stats.update(loss.detach().item(), model_output.detach(), labels)

        return stats.compute()

    def __eval(self) -> dict[str, float]:
        self.model.eval()
        stats = Stats()
        with torch.no_grad():
            for batch in tqdm(self.val_dataloader):
                data, labels = batch
                data = data.to(self.device, dtype=torch.float32)
                labels = labels.to(self.device, dtype=torch.int64)

                with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                    model_output = self.model(pixel_values=data)
                    loss = self.loss_fn(model_output, labels)

                stats.update(loss.detach().item(), model_output.detach(), labels)

        return stats.compute()

In [12]:
ENCODER = "mvit"  # one of: "videomae_base", "mvit"

if ENCODER == "videomae_base":
    processor, _ = load_videomae_v2("OpenGVLab/VideoMAEv2-Base")
    model = load_model()
elif ENCODER == "mvit":
    model, processor = load_model_mvit()
 
train_ds = ASLCitizen(
    "data/splits/train.csv",
    data_dir=f"{DATASET}/videos",
    image_processor=processor,
)
val_ds = ASLCitizen(
    "data/splits/val.csv",
    data_dir=f"{DATASET}/videos",
    image_processor=processor,
)
 
train_dataloader = DataLoader(
    train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=4
)
val_dataloader = DataLoader(
    val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=4
)
 
loss_fn = torch.nn.CrossEntropyLoss()

trainer = LPFTTrainer(
    model, loss_fn, train_dataloader, val_dataloader,
    checkpoint_path="out/best_model.pt",
    checkpoint_metric="recall@1",
)

history = trainer.run(
    lp_epochs=5, 
    lp_lr=1e-3,
    ft_epochs=50, 
    ft_lr=2e-5,
    head_lr_in_ft=1e-4,
)

# afterwards, restore the best epoch's weights for eval/inference:
best_ckpt = trainer.load_best_checkpoint()
print(best_ckpt["phase"], best_ckpt["epoch"], best_ckpt["val_stats"])

Downloading: "https://download.pytorch.org/models/mvit_v2_s-ae3be167.pth" to /root/.cache/torch/hub/checkpoints/mvit_v2_s-ae3be167.pth


100%|██████████| 132M/132M [00:00<00:00, 215MB/s] 


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/best_model.pt'

In [ ]:
!pip install git+https://github.com/ml-jku/hopfield-layers